# 🔗 Python Union Find — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Union Find is like a club system in a school. Every student starts in their own solo club.
> When two students become friends, their clubs merge — one club adopts the other.
> To check if any two students are in the same club, you just ask: who's the president?
> If they share the same president, they're in the same club. Merging and checking are nearly O(1).

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Union Find? The Visual Model](#1) |
| 2 | [Creating / Setup](#2) |
| 3 | [The Core API — All Operations](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Number of Connected Components (LC 323)](#5) |
| 6 | [Pattern 2: Number of Provinces (LC 547)](#6) |
| 7 | [Pattern 3: Redundant Connection (LC 684)](#7) |
| 8 | [Pattern 4: Accounts Merge (LC 721)](#8) |
| 9 | [The Union Find Decision Map](#9) |
| 10 | [Interview Cheat Sheet](#10) |

<a id='1'></a>
## 1. What Is Union Find? The Visual Model

```
               UNION FIND — THE CLUB SYSTEM

  Initially: every node is its own club president
  parent = [0, 1, 2, 3, 4]  (node i's president is i)
  rank   = [0, 0, 0, 0, 0]  (all clubs start equal size)

  union(0, 1) → clubs 0 and 1 merge:
  parent = [0, 0, 2, 3, 4]  (1's president is now 0)

  union(2, 3) → clubs 2 and 3 merge:
  parent = [0, 0, 2, 2, 4]  (3's president is now 2)

  union(0, 2) → two big clubs merge:
  parent = [0, 0, 0, 2, 4]  (2's president becomes 0)

  find(3) → 3→2→0  → president is 0
  find(1) → 1→0    → president is 0
  same_club? YES  (both report to president 0)

  PATH COMPRESSION: while finding root, flatten the tree:
    3→2→0  becomes  3→0  (skip intermediaries on the way back)

  UNION BY RANK: always attach shorter tree under taller:
    prevents worst-case O(n) chains, keeps tree nearly flat

  With both optimizations: find() ≈ O(α(n)) — practically O(1)
  α = inverse Ackermann function, grows so slowly it's ≤ 4 for all practical n
```

<a id='2'></a>
## 2. Creating / Setup

In [ ]:
# UnionFind class — reusable across all problems
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))   # each node starts as its own president
        self.rank   = [0] * n          # rank = upper bound on tree height
        self.count  = n                # number of distinct components

    def find(self, x):
        # path compression: point every node directly to root while finding
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # flatten on the way back
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)  # find presidents of both clubs
        if rx == ry:
            return False              # already in the same club — no merge needed
        # union by rank: attach shorter tree under taller tree
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx           # ensure rx has the higher rank
        self.parent[ry] = rx          # ry club joins rx club
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1        # equal height → new root gets one taller
        self.count -= 1               # one fewer distinct component
        return True                   # merge happened

    def connected(self, x, y):
        return self.find(x) == self.find(y)  # same president = same component

# Demo with 5 nodes
uf = UnionFind(5)
print("initial components:", uf.count)    # 5
uf.union(0, 1)
uf.union(2, 3)
print("after union(0,1) and union(2,3):", uf.count)   # 3
uf.union(0, 2)
print("after union(0,2):", uf.count)      # 2
print("connected(1, 3):", uf.connected(1, 3))  # True
print("connected(1, 4):", uf.connected(1, 4))  # False
print("UnionFind class loaded.")

<a id='3'></a>
## 3. The Core API — All Operations

```
OPERATION           COMPLEXITY       WHAT IT DOES
──────────────────────────────────────────────────────────────────
find(x)             O(α(n)) ≈ O(1)   return root/president of x's component
union(x, y)         O(α(n)) ≈ O(1)   merge two components; return False if already same
connected(x, y)     O(α(n)) ≈ O(1)   True if x and y share the same root
uf.count            O(1)             current number of distinct components

OPTIMIZATIONS (both together = inverse Ackermann time):
  Path Compression:  during find(), set parent[x] = root directly
  Union by Rank:     always attach shorter tree under taller tree
  Union by Size:     alternative — attach smaller count under larger count

THINGS YOU DO NOT DO:
❌  Union without find first — always find roots before comparing
❌  Use Union Find for directed graphs — UF is for undirected components only
❌  Forget to decrement count in union() if you need component count
❌  Use recursion for find() on very large n (stack overflow) — prefer iterative
❌  Union Find for shortest path — wrong tool; use BFS/Dijkstra
```

In [ ]:
# Iterative find — avoids recursion depth issues on large inputs
class UnionFindIterative:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank   = [0] * n
        self.count  = n

    def find(self, x):
        root = x
        while self.parent[root] != root:  # walk up to find the root
            root = self.parent[root]
        while self.parent[x] != root:     # second pass: compress path
            nxt = self.parent[x]
            self.parent[x] = root          # flatten: point directly to root
            x = nxt
        return root

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        self.count -= 1
        return True

    def connected(self, x, y):
        return self.find(x) == self.find(y)

# Demo: build incrementally and verify state
ufi = UnionFindIterative(6)
edges = [(0,1),(1,2),(3,4)]
for u, v in edges:
    ufi.union(u, v)
print("components after (0-1),(1-2),(3-4):", ufi.count)    # 3
print("connected(0,2):", ufi.connected(0, 2))  # True (0-1-2)
print("connected(0,3):", ufi.connected(0, 3))  # False
print("Iterative UnionFind demonstrated.")

<a id='4'></a>
## 4. Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                   WHAT TO DO
──────────────────────────────────────────────────────────────────────
"connected components" (undirected)     UnionFind — count = answer
"same group / same set"                 connected(x, y) query
"merge accounts / emails"               UF over string keys (dict-based)
"detect cycle in undirected graph"      union returns False = already connected = cycle
"redundant edge in undirected graph"    first edge whose union() returns False
"minimum spanning tree (Kruskal)"       sort edges by weight, union until n-1 edges
"dynamic connectivity (edges arrive)"  UF — handle each new edge with union()
"how many distinct groups"              uf.count after all unions
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Number of Connected Components — LC 323

---

```
PROBLEM:
  Given n nodes (0 to n-1) and a list of undirected edges, count the number
  of connected components in the graph.

TRICK:
  Initialize UnionFind with n components. Process each edge with union().
  After all edges, uf.count is the answer — each successful merge reduces it by 1.

SLOW MOTION TRACE on n=5, edges=[[0,1],[1,2],[3,4]]:

  Initial: parent=[0,1,2,3,4], count=5

  union(0,1): roots 0,1 differ → merge → parent=[0,0,2,3,4], count=4
  union(1,2): find(1)→0, find(2)→2 → merge → parent=[0,0,0,3,4], count=3
  union(3,4): roots 3,4 differ → merge → parent=[0,0,0,3,3], count=2

  answer = uf.count = 2
  Components: {0,1,2} and {3,4}

KEY INSIGHT:
  Start with n components. Every successful union() reduces count by exactly 1.
  Final count = number of components.

TIME:  O(E * α(n)) ≈ O(E) — E = number of edges
SPACE: O(n) — parent and rank arrays
```

In [ ]:
def count_components(n, edges):
    """
    LC 323 — Number of Connected Components in an Undirected Graph
    Approach: UnionFind; start at n components, decrement per successful merge.
    Args:
        n (int): number of nodes labeled 0 to n-1.
        edges (List[List[int]]): undirected edges [u, v].
    Returns:
        int: number of connected components.
    Time:  O(E * α(n)) ≈ O(E) — nearly linear in number of edges
    Space: O(n) — parent and rank arrays of size n
    """
    uf = UnionFind(n)          # each node starts as its own club
    for u, v in edges:
        uf.union(u, v)         # merge the clubs containing u and v
    return uf.count            # remaining clubs = connected components

# Slow motion on n=5, edges=[[0,1],[1,2],[3,4]]:
# initial count=5
# union(0,1) → count=4
# union(1,2) → find(1)=0, find(2)=2 → count=3
# union(3,4) → count=2
# return 2

def test_harness(fn):
    tests = [
        (5, [[0,1],[1,2],[3,4]], 2),         # {0,1,2} and {3,4}
        (5, [[0,1],[1,2],[2,3],[3,4]], 1),   # all connected
        (5, [], 5),                           # no edges → 5 isolated nodes
        (3, [[0,1],[1,2],[0,2]], 1),          # triangle → 1 component
        (4, [[0,1],[2,3]], 2),                # two pairs
        (1, [], 1),                           # single node
    ]
    passed = 0
    for *inputs, expected in tests:
        n, edges = inputs
        got = fn(n, edges)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | n={n} edges={edges} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(count_components)
print("count_components defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Number of Provinces — LC 547

---

```
PROBLEM:
  Given an n×n adjacency matrix isConnected where isConnected[i][j]=1 means
  cities i and j are directly connected, return the number of provinces
  (a province is a group of directly or indirectly connected cities).

TRICK:
  Same as LC 323 but input is a matrix instead of an edge list.
  Only scan the upper triangle (j > i) to avoid double-processing symmetric edges.
  Union each pair where isConnected[i][j] == 1.

SLOW MOTION TRACE on isConnected=[[1,1,0],[1,1,0],[0,0,1]]:

  n=3, initial count=3
  i=0: j=1, isConnected[0][1]=1 → union(0,1) → count=2
  i=0: j=2, isConnected[0][2]=0 → skip
  i=1: j=2, isConnected[1][2]=0 → skip
  answer = 2 (province {0,1} and province {2})

KEY INSIGHT:
  Matrix input → iterate upper triangle. Logic identical to edge-list version.
  The diagonal (i==j) is always 1 (city connected to itself) — skip it.

TIME:  O(n²) — must scan the upper triangle of the n×n matrix
SPACE: O(n)  — UnionFind arrays
```

In [ ]:
def find_circle_num(is_connected):
    """
    LC 547 — Number of Provinces
    Approach: UnionFind on adjacency matrix; scan upper triangle for edges.
    Args:
        is_connected (List[List[int]]): n×n symmetric adjacency matrix.
    Returns:
        int: number of provinces (connected components).
    Time:  O(n²) — scan upper triangle of n×n matrix
    Space: O(n)  — UnionFind parent and rank arrays
    """
    n = len(is_connected)
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i + 1, n):         # upper triangle only — matrix is symmetric
            if is_connected[i][j] == 1:
                uf.union(i, j)             # cities i and j are in the same province
    return uf.count

# Slow motion on [[1,1,0],[1,1,0],[0,0,1]]:
# i=0,j=1: connected → union(0,1), count=2
# i=0,j=2: not connected → skip
# i=1,j=2: not connected → skip
# return 2

def test_harness(fn):
    tests = [
        ([[1,1,0],[1,1,0],[0,0,1]], 2),           # {0,1} and {2}
        ([[1,0,0],[0,1,0],[0,0,1]], 3),           # all isolated
        ([[1,1,1],[1,1,1],[1,1,1]], 1),           # all connected
        ([[1,0,0,1],[0,1,1,0],[0,1,1,0],[1,0,0,1]], 2),  # {0,3} and {1,2}
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_circle_num)
print("find_circle_num defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Redundant Connection — LC 684

---

```
PROBLEM:
  Given edges forming a tree plus one extra edge, find the redundant edge
  that creates a cycle. Return it.

TRICK:
  Process edges in order. For each edge [u, v], try to union u and v.
  If they are ALREADY connected (same component), this edge creates a cycle — return it.
  The first edge where union() returns False is the redundant one.

SLOW MOTION TRACE on edges=[[1,2],[1,3],[2,3]]:

  n=3 (nodes labeled 1..3), UF for nodes 0..3 (index 0 unused)

  edge [1,2]: find(1)=1, find(2)=2 → different → union → count=3 (no cycle)
  edge [1,3]: find(1)=1, find(3)=3 → different → union → count=2 (no cycle)
  edge [2,3]: find(2)=find(1)=1, find(3)=find(1)=1 → SAME → cycle!
  return [2,3]

KEY INSIGHT:
  In a tree, any edge connecting two already-connected nodes creates a cycle.
  Union Find detects this instantly: union() returns False = cycle = redundant edge.

TIME:  O(E * α(n)) ≈ O(E)
SPACE: O(n)
```

In [ ]:
def find_redundant_connection(edges):
    """
    LC 684 — Redundant Connection
    Approach: Process edges one by one; the first edge where union() fails creates the cycle.
    Args:
        edges (List[List[int]]): list of undirected edges [u, v] with nodes 1..n.
    Returns:
        List[int]: the redundant edge [u, v] that creates a cycle.
    Time:  O(E * α(n)) ≈ O(E) — one union per edge
    Space: O(n) — UnionFind arrays
    """
    n = len(edges)             # nodes labeled 1..n, exactly n edges
    uf = UnionFind(n + 1)      # +1 so index 1..n are valid (index 0 unused)
    for u, v in edges:
        if not uf.union(u, v): # already in same component → this edge is redundant
            return [u, v]
    return []                  # should not reach here per problem constraints

# Slow motion on [[1,2],[1,3],[2,3]]:
# union(1,2): different roots → merge, return True
# union(1,3): different roots → merge, return True
# union(2,3): find(2)=root shared with find(3) → return False → [2,3]

def test_harness(fn):
    tests = [
        ([[1,2],[1,3],[2,3]], [2,3]),
        ([[1,2],[2,3],[3,4],[1,4],[1,5]], [1,4]),
        ([[1,2],[2,3],[3,1]], [3,1]),
        ([[1,4],[3,4],[1,3],[1,2],[4,5]], [1,3]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(inputs[0])
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | edges={inputs[0]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(find_redundant_connection)
print("find_redundant_connection defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: Accounts Merge — LC 721

---

```
PROBLEM:
  Given a list of accounts where each account is [name, email1, email2, ...],
  merge accounts that share at least one email. Return merged accounts with
  emails sorted and the account owner's name.

TRICK:
  UnionFind over EMAILS (not account indices).
  Map each email to an integer index. Union all emails within the same account.
  Also map each email to its owner's name.
  After all unions, group emails by their root representative.
  Sort each group and prepend the owner name.

SLOW MOTION TRACE on accounts:
  [["John","a@x","b@x"],["John","c@x"],["Mary","b@x"]]

  email → index: {a@x:0, b@x:1, c@x:2}
  account 0 (John): union(a@x, b@x)  → merge 0 and 1
  account 1 (John): c@x alone, no union
  account 2 (Mary): union(b@x)=union(1) — b@x already in group 0
    → b@x and a@x already merged; Mary's b@x joins that group

  Wait — account 2 has only b@x, no union needed (single email).
  But b@x was already unioned with a@x in account 0.
  Group by root: root of 0,1 → {a@x, b@x} owner=John; root of 2 → {c@x} owner=John

KEY INSIGHT:
  Union emails within each account (first email = anchor for the rest).
  Group by root to collect final merged sets.

TIME:  O(N * K * α(N*K)) where N=accounts, K=max emails per account
SPACE: O(N*K) — email-to-index map
```

In [ ]:
from collections import defaultdict

def accounts_merge(accounts):
    """
    LC 721 — Accounts Merge
    Approach: Union emails within each account; group by root; sort and format results.
    Args:
        accounts (List[List[str]]): each account = [name, email1, email2, ...].
    Returns:
        List[List[str]]: merged accounts, each = [name, sorted_email1, ...].
    Time:  O(N*K*α(N*K)) — N accounts, K emails each, union-find ops per email
    Space: O(N*K) — email-to-index and email-to-owner maps
    """
    email_to_idx  = {}    # email string → integer index for UnionFind
    email_to_name = {}    # email string → account owner name

    # assign a unique index to every email we've seen
    for account in accounts:
        name = account[0]
        for email in account[1:]:
            if email not in email_to_idx:
                email_to_idx[email] = len(email_to_idx)  # next available index
            email_to_name[email] = name                   # record owner

    uf = UnionFind(len(email_to_idx))  # one slot per unique email

    # union all emails within the same account (use first email as anchor)
    for account in accounts:
        first_idx = email_to_idx[account[1]]             # anchor = first email
        for email in account[2:]:                        # union rest to the anchor
            uf.union(first_idx, email_to_idx[email])

    # group emails by their root representative
    root_to_emails = defaultdict(list)
    for email, idx in email_to_idx.items():
        root = uf.find(idx)                              # which club does this email belong to?
        root_to_emails[root].append(email)

    # build result: [name] + sorted emails for each group
    result = []
    for root, emails in root_to_emails.items():
        name = email_to_name[emails[0]]                  # any email → owner name
        result.append([name] + sorted(emails))
    return result

# Slow motion on [["John","a@x","b@x"],["John","c@x"],["Mary","b@x"]]:
# email_to_idx: {a@x:0, b@x:1, c@x:2}
# account 0: anchor=0(a@x), union(0,1) → a@x and b@x in same group
# account 1: no pair to union (only 1 email c@x)
# account 2 (Mary has b@x): anchor=1(b@x), no second email → no union
#   but a@x and b@x are already merged; Mary's b@x is in John's merged group
# root_to_emails: root_of_0 → [a@x, b@x], root_of_2 → [c@x]
# result: [["John", "a@x", "b@x"], ["John", "c@x"]]  (Mary merged with John)

def test_harness(fn):
    def normalize(result):
        # sort each inner list after the first element, then sort the whole result
        return sorted([row[0:1] + sorted(row[1:]) for row in result])

    tests = [
        (
            [["John","j@g","j@m"],["John","j@g2"],["Mary","m@g"]],
            [["John","j@g","j@m"],["John","j@g2"],["Mary","m@g"]]
        ),
        (
            [["Gabe","Gabe0@m"],["Kevin","Kevin0@m"],["Ethan","Ethan0@m"],
             ["Hanzo","Hanzo0@m"],["Fern","Fern0@m"]],
            [["Gabe","Gabe0@m"],["Kevin","Kevin0@m"],["Ethan","Ethan0@m"],
             ["Hanzo","Hanzo0@m"],["Fern","Fern0@m"]]
        ),
        (
            [["A","a@x","b@x"],["B","b@x","c@x"]],
            [["A","a@x","b@x","c@x"]]  # all merged via shared b@x
        ),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = normalize(fn(inputs[0]))
        exp = normalize(expected)
        status = "PASSED" if got == exp else "FAILED"
        if status == "FAILED":
            print(f"{status} | expected={exp} | got={got}")
        passed += (got == exp)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(accounts_merge)
print("accounts_merge defined.")

<a id='9'></a>
## 9. The Union Find Decision Map

```
QUESTION TYPE                         KEY TECHNIQUE                 LC PROBLEMS
───────────────────────────────────────────────────────────────────────────────
Count connected components            UF init n, count after edges   323
Province / circle count               UF on matrix upper triangle    547
Redundant edge / cycle in undirected  First union() → False          684
Merge accounts / sets by key          UF over string→index map       721
Minimum spanning tree (Kruskal)       Sort edges, union until n-1    1584
Dynamic connectivity                  Union on arrival, query anytime —
Smallest component with property      UF + size tracking             —

CHOOSE UNION FIND WHEN:
  ✅ Graph is undirected
  ✅ Queries are "same group?" or "how many groups?"
  ✅ Edges arrive dynamically (online algorithm)
  ✅ You need O(1) merge and O(1) lookup

DO NOT USE UNION FIND WHEN:
  ❌ Graph is directed (use DFS/BFS/Topo sort)
  ❌ You need shortest path (use BFS/Dijkstra)
  ❌ Edges can be REMOVED (UF is append-only)
```

<a id='10'></a>
## 10. Interview Cheat Sheet

**1. When to reach for Union Find:**

| Signal | What to Do |
|--------|------------|
| "connected components" (undirected) | UF — uf.count is the answer |
| "same group / province" | uf.connected(u, v) |
| "redundant / cycle-creating edge" | first edge where union() returns False |
| "merge groups by shared element" | UF over element → index map |
| "minimum spanning tree" | Sort edges + Kruskal's (UF) |

**2. The O(α(n)) operations — memorize these:**

```python
# FIND with path compression (recursive)
def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])  # flatten on the way back
    return parent[x]

# UNION by rank
def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return False        # already same group
    if rank[rx] < rank[ry]: rx, ry = ry, rx
    parent[ry] = rx
    if rank[rx] == rank[ry]: rank[rx] += 1
    return True
```

**3. Common templates:**

```python
# TEMPLATE: COUNT COMPONENTS (edge list)
uf = UnionFind(n)
for u, v in edges:
    uf.union(u, v)
return uf.count

# TEMPLATE: REDUNDANT EDGE
uf = UnionFind(n + 1)
for u, v in edges:
    if not uf.union(u, v):
        return [u, v]

# TEMPLATE: STRING KEYS (accounts merge pattern)
email_to_id = {}
for account in accounts:
    for email in account[1:]:
        if email not in email_to_id:
            email_to_id[email] = len(email_to_id)
uf = UnionFind(len(email_to_id))
for account in accounts:
    anchor = email_to_id[account[1]]
    for email in account[2:]:
        uf.union(anchor, email_to_id[email])
```

**4. Gotchas to not forget:**

```
❌  Applying UF to directed graphs — only works for undirected
❌  Forgetting to decrement count in union() if you need component count
❌  Not doing path compression — find() degrades to O(n) without it
❌  Nodes labeled 1..n → initialize UnionFind(n+1) to avoid off-by-one
✅  union() returning False = cycle detected / nodes already connected
✅  Use a dict for string keys: string → integer index → UnionFind
✅  Path compression + union by rank = inverse Ackermann ≈ O(1) per op
✅  uf.count starts at n; each successful union decrements it
```

## Summary Map

```
                    🔗 UNION FIND
                         │
           ┌─────────────┼─────────────┐
           │             │             │
        find()        union()       connected()
     path compress   union by rank   find(x)==find(y)
     O(α(n))≈O(1)    O(α(n))≈O(1)   O(α(n))≈O(1)
           │             │
     ┌─────┴──────┐  ┌───┴──────────┐
     │            │  │              │
  COUNT        QUERY  RETURNS       RETURNS
  COMPONENTS   SAME   True=merge    False=cycle!
  uf.count     GROUP  happened      redundant edge
  LC 323       LC 547               LC 684
                  │
           STRING KEYS
           email→int→UF
           LC 721

CORE RULE:
  Find the root (president) of each element.
  Same root = same group. Different roots = merge them.
  union() returns False = already same group = cycle or redundant edge.
```

---
*End of Union Find Master Guide — Sean Edition*